In [1]:
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
	
# Set such that PDF fonts export in a manner that they
# are editable in illustrator/affinity
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# set to define axes linewidths
matplotlib.rcParams['axes.linewidth'] = 0.5

# this defines some prefactors so inline figures look nice
%matplotlib inline
%config InlineBackend.figure_format='retina'


matplotlib.rc('font', **font)

from tqdm import tqdm
import pickle
from sparrow import Protein
import protfasta
from tqdm.auto import tqdm

In [2]:
from shephard.apis import uniprot
from shephard.interfaces import si_domains, si_protein_attributes

from finches import Mpipi_frontend, CALVADOS_frontend
mf = Mpipi_frontend()
cf = CALVADOS_frontend()

/home/wenyuantong/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/finches/forcefields/calvados.py:236: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.038286503882254706' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  r.loc['H','q'] = 1. / ( 1 + 10**(self.pH-6) )


In [ ]:
#HP = uniprot.uniprot_fasta_to_proteome('/home/wenyuantong/Desktop/data/PsyTEC_2_new.fasta')
#529

In [3]:
#HP1 = uniprot.uniprot_fasta_to_proteome('/home/wenyuantong/Desktop/data/PsyTEC_2_new.fasta')
#529

In [4]:
#HP2 = uniprot.uniprot_fasta_to_proteome('/home/wenyuantong/Desktop/data/UP000006548_3702.fasta') 
#27448

In [3]:
HP = uniprot.uniprot_fasta_to_proteome('/home/wenyuantong/Desktop/data/combined.fasta')
#27977

In [4]:
si_domains.add_domains_from_file(HP, '/home/wenyuantong/Desktop/data/IDR_psytec_domain.tsv', skip_bad=True, safe=False)

In [5]:
si_domains.add_domains_from_file(HP, '/home/wenyuantong/Desktop/data/IDR_arabi_domain.tsv', skip_bad=True, safe=False)

In [14]:
# build set of domains to work with - default in paper is IDRs between 100 and 150 inclusive
LOWER = 951
UPPER = 1000
size_d = []
for d in HP.domains:
    #display(d)
    if len(d) >= LOWER and len(d) <= UPPER:
        size_d.append(d)

# construct name-to-sequence mapping
name2seq = {}
for d1 in size_d:
    d1_name = d1.protein.unique_ID + "_" + d1.domain_name
    name2seq[d1_name] = d1.sequence


# construct name-to-sequence mapping
name2domain = {}
for d1 in size_d:
    d1_name = d1.protein.unique_ID + "_" + d1.domain_name
    name2domain[d1_name] = d1

print(f"Found {len(size_d)} IDRs; this will be {len(size_d)**2} epsilon calculations")

Found 13 IDRs; this will be 169 epsilon calculations


In [15]:
mode = 'mpipi'

In [16]:
# if set to True we recompute from scratch and save the newly computed values
# to the pickle files, if False, we read from recomputed pickle files
RECOMPUTE = True

if mode == 'mpipi':
    if RECOMPUTE:
        cross_interaction = {}
        
        for d1 in tqdm(size_d):
            
            d1_name = d1.protein.unique_ID + "_" + d1.domain_name
            #print(d1_name)
            cross_interaction[d1_name] = {}        
                
            for d2 in size_d:
                d2_name = d2.protein.unique_ID + "_" + d2.domain_name
                
                cross_interaction[d1_name][d2_name] = mf.epsilon(d1.sequence, d2.sequence)
                
            with open('/home/wenyuantong/Desktop/data/15005027/finches/figure_4/cross_interaction_mpipi.pkl', 'wb') as file:
            # Use pickle to serialize the dictionary and write it to the file
                pickle.dump(cross_interaction, file)
    else:
        # load
    
        with open('/home/wenyuantong/Desktop/data/15005027/finches/figure_4/cross_interaction_mpipi.pkl', 'rb') as file:
            # Load the dictionary back from the pickle file
            cross_interaction = pickle.load(file)



  0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
# save cross_interaction as pd df

In [17]:
import pandas as pd
# Create a DataFrame directly from the nested dictionary
df = pd.DataFrame(cross_interaction)
df.to_csv("/home/wenyuantong/Desktop/data/arabi_cross/951_1000.csv", index=True)

In [20]:
df1 = pd.read_csv("/home/wenyuantong/Desktop/data/arabi_cross/951_1000.csv", index_col=0)

In [21]:
df1

,Q8LK56_N-terminal_1_993,F4I1T7_C-terminal_868_1819,O64768_N-terminal_1_959,F4I6U7_IDP_1_957,F4IDB4_C-terminal_204_1163,Q9FLQ7_middle_339_1310,F4JCZ1_IDP_1_957,Q2LAE1_N-terminal_1_965,Q56YR0_IDP_1_991,Q9FMM3_middle_589_1570,F4JC20_IDP_1_997,F4J0W8_IDP_1_978,F4K5K6_C-terminal_199_1164
Q8LK56_N-terminal_1_993,24.086605,25.810694,18.490179,17.376799,28.286302,22.396411,-5.428971,28.437447,18.115196,27.584183,6.604286,24.155511,27.839085
F4I1T7_C-terminal_868_1819,26.922289,29.239742,23.697377,19.897064,31.961577,27.181042,-4.310159,34.962947,20.608792,32.340643,7.402286,25.436436,35.868695
O64768_N-terminal_1_959,19.145723,23.524403,22.774353,16.790310,29.198760,19.779609,-14.748184,35.902560,17.262934,26.256566,10.835584,16.012512,38.044841
F4I6U7_IDP_1_957,18.030472,19.793108,16.825399,12.900019,25.139013,16.154001,-15.091493,27.952360,13.756085,23.548667,4.812267,17.107450,28.948544
F4IDB4_C-terminal_204_1163,29.258644,31.695231,29.168344,25.060454,36.563304,27.655863,-1.096754,39.328353,26.125642,34.883222,18.977634,28.713006,40.561434
Q9FLQ7_middle_339_1310,22.880285,26.621761,19.515067,15.904711,27.314433,25.976739,-7.057549,30.871554,16.898164,28.756806,1.186412,22.211760,32.302837
F4JCZ1_IDP_1_957,-5.633196,-4.287640,-14.779006,-15.091493,-1.100192,-7.168169,-39.383790,-2.593296,-15.847260,-1.229802,-34.022608,-4.207559,-2.947923
Q2LAE1_N-terminal_1_965,29.262575,34.491944,35.679332,27.720631,39.124580,31.095492,-2.571797,47.979573,28.793003,37.342253,23.702708,25.334511,51.406953
Q56YR0_IDP_1_991,18.151756,19.797750,16.705503,13.284130,25.308392,16.574183,-15.303560,28.037586,13.846785,23.320516,4.999094,16.921638,28.895420
Q9FMM3_middle_589_1570,27.893170,31.352640,25.641596,22.949159,34.101724,28.463967,-1.198494,36.695798,23.534247,32.838014,13.095491,26.902306,36.928954


In [20]:
df_psy = pd.read_csv('/home/wenyuantong/Desktop/data/IDR_psytec_domain.tsv', sep='\t', header=None)

In [38]:
df_psy[0] = df_psy[0].str.replace(' ', '')
df_psy

,0,1,2,3
0,HopX1a,1,88,N-terminal
1,HopX1a,200,247,Middle
2,HopX1a,298,380,C-terminal
3,HopX1c,1,87,N-terminal
4,HopX1c,212,267,Middle
...,...,...,...,...
588,HopZ5a,1,44,N-terminal
589,HopZ5a,252,346,C-terminal
590,HopZ5b,1,44,N-terminal
591,HopZ5b,283,346,C-terminal


In [39]:
unique_names = df_psy[0].unique()

In [53]:
#unique_names

In [41]:
import re 
search_pattern = '|'.join([re.escape(s) for s in unique_names])

In [117]:
# Identify columns to drop based on partial match
columns_to_drop_partial = df1.columns[df1.columns.str.contains(search_pattern, case=False, regex=True)].tolist()

In [118]:
# Identify rows to drop based on partial match
rows_to_drop_partial = df1.index[df1.index.str.contains(search_pattern, case=False, regex=True)].tolist()

In [119]:
# Drop columns matching the labels
#df_temp = df1.drop(columns=[label for label in unique_names if label in df1.columns])


# Drop identified columns
df_temp_partial = df1.drop(columns=columns_to_drop_partial)

In [120]:
# Drop rows matching the labels from the temporary DataFrame
#df_filtered = df_temp.drop(index=[label for label in unique_names if label in df1.index])

df_filtered_partial = df_temp_partial.drop(index=rows_to_drop_partial)

In [121]:
#print(f"\nSubstrings to match and drop: {unique_names}")
#print(f"Columns identified for dropping: {columns_to_drop_partial}")
#print(f"Rows identified for dropping: {rows_to_drop_partial}")
#print(f"\nDataFrame after dropping columns and rows with partial label matches:")

In [122]:
#print(f"\nDataFrame after dropping labels: {unique_names}")
display(df_filtered_partial)

,O64855_C-terminal_75_493,Q9C660_N-terminal_1_414,Q9SHI1_N-terminal_1_413,Q95749_N-terminal_1_443,Q9FYE2_IDP_1_406,F4J2C6_C-terminal_629_1078,F4K2E9_N-terminal_1_420,F4KE50_N-terminal_1_417,Q9SST1_N-terminal_1_427,Q9ZQ70_N-terminal_1_407,...,Q9FMZ3_middle_209_641,F4HRT5_C-terminal_721_1132,F4HZ48_N-terminal_1_407,Q9LTX1_middle_95_501,Q9FPW4_N-terminal_1_434,Q9LF34_IDP_1_402,F4ICK6_middle_482_919,Q9FLK7_N-terminal_1_425,O23088_IDP_1_431,Q9FFP2_C-terminal_72_492
O64855_C-terminal_75_493,11.091540,7.112538,15.403322,7.384183,8.553026,14.053506,6.474017,8.541504,6.262409,5.081479,...,16.075310,10.060828,0.831343,10.612154,1.985776,9.136183,16.103520,22.695542,8.538303,11.076905
Q9C660_N-terminal_1_414,7.198438,23.100350,21.128956,3.330235,14.241246,21.520194,7.619287,16.766619,7.427799,12.054206,...,18.210178,19.470347,10.622791,14.710131,9.297089,12.615274,18.589605,28.020686,20.316435,12.150448
Q9SHI1_N-terminal_1_413,15.627099,21.180115,25.328773,15.011031,16.910351,23.604275,11.679536,14.573651,11.124441,13.740556,...,23.562501,17.792709,6.229495,18.816952,13.146371,15.431704,28.069357,33.331721,21.089178,18.951607
Q95749_N-terminal_1_443,6.984137,3.112228,13.994483,5.488136,3.146256,9.407457,-1.684081,-1.974034,-2.730752,-1.288026,...,12.799345,2.386594,-12.792394,6.644378,-2.703752,2.630970,18.049144,23.481590,4.639534,7.512503
Q9FYE2_IDP_1_406,8.826892,14.521862,17.201908,3.432984,10.412197,16.863826,6.576273,12.068955,6.150035,8.041652,...,15.950284,14.478912,5.164683,11.910122,5.182881,10.324708,16.744994,24.557451,13.874953,10.561077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9LF34_IDP_1_402,9.522539,12.991849,15.853965,2.899302,10.427441,16.428751,8.312144,13.906560,8.291059,8.450479,...,15.626539,15.200349,8.107258,11.519484,4.804474,11.164580,13.839323,22.301864,12.385607,10.206952
F4ICK6_middle_482_919,15.404965,17.570996,26.467225,18.255184,15.521615,22.417435,7.982190,9.314492,7.378091,10.684187,...,24.845281,13.307445,-0.971141,18.039310,10.373167,12.701844,30.227079,35.422086,18.770696,18.928034
Q9FLK7_N-terminal_1_425,22.375134,27.295445,32.390590,24.476105,23.459589,30.328489,17.730244,19.264186,17.327642,19.866538,...,30.781512,22.481502,10.462589,25.307051,20.569192,21.094939,36.505585,40.482803,27.577024,26.392589
O23088_IDP_1_431,8.300578,19.515091,20.208424,4.768709,13.070141,19.690498,7.236577,14.155681,7.067262,10.622482,...,17.677906,16.752223,7.437771,13.946020,8.654529,11.552237,19.075556,27.193121,17.776851,12.129098


In [123]:
df_filtered_partial.to_csv("/home/wenyuantong/Desktop/data/arabi_cross/401_450.csv", index=True)